In [ ]:
import faiss 
from sentence_transformers import SentenceTransformer


In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

sentences = [
    "The patient has high fever and cough.",
    "Ayushman Bharat covers hospitalization costs.",
    "The stock market crashed yesterday.",
    "Diabetes management requires diet control.",
    "FAISS is used for similarity search."
]
embeddings = model.encode(sentences) 
print(embeddings.shape)

In [ ]:
index = faiss.IndexFlatL2(384)

In [ ]:
index.add(embeddings)
index.ntotal

In [ ]:
query = "what diseases does ayushman bharat cover"
query_embeddings = model.encode(query)
print(query_embeddings.shape)
query_embeddings = query_embeddings.reshape(1,384)  
distances, indices = index.search(query_embeddings, k = 3)
#for i in sentences:
  #  print(indices[i])
for i in indices[0]:
    print(sentences[i])


In [ ]:
from groq import Groq 
client = Groq(api_key = "your api key")

response = client.chat.completions.create(
    model = "llama-3.1-8b-instant",
    messages = [{"role": "user", "content": "what is ayushman bharat in one sentence"}],
    max_tokens = 100
)
print(response.choices[0].message.content)

In [ ]:
context = " ".join([sentences[i] for i in indices[0]])
query = "how do you manage diabetes"
prompt = f"Context: {context}\nQuestion: {query}\nAnswer based only on the context above."



In [ ]:
response = client.chat.completions.create(
    model = "llama-3.1-8b-instant",
    messages = [{"role": "user", "content":prompt}],
    max_tokens = 100
)
print(response.choices[0].message.content)

In [ ]:
import fitz 

read_pdf = fitz.open(r"C:\Users\Intro\Downloads\INSIGHTS-PT-2026-Exclusive-Government-Schemes.pdf")



In [ ]:
store = ""
for reading in read_pdf:
    store += reading.get_text()

len(store)
print(store[90000:95000])
    

In [ ]:
#store = 221k characters 
#slide window to 1500 then move forward by 1300 
list_store = []
for i in range(0 , len(store), 1300):
    chunk = store[i : i + 1500]
    list_store.append(chunk)
    
len(list_store)
    

In [ ]:
rag_embeddings = model.encode(list_store)
print(rag_embeddings.shape)

In [ ]:
faiss.normalize_L2(rag_embeddings)


ragindex = faiss.IndexFlatIP(384)
ragindex.add(rag_embeddings)
ragindex.ntotal

In [ ]:
def ask(query):
    queryembeddings = model.encode(query)
    queryembeddings = queryembeddings.reshape(1,384)
    faiss.normalize_L2(queryembeddings)
    
    distances, indices = ragindex.search(queryembeddings, k = 4)
    context = " ".join([list_store[i] for i in indices[0]])
    prompt = f"context: {context} \nQuestion: {query} \nAnswer based only on the context above, if you can't find information then just answer I cannot find in the document"
    response_rag = client.chat.completions.create(
        model = "llama-3.1-8b-instant",
        messages = [{"role":"user", "content":prompt}],
        max_tokens = 2000

    )
    return response_rag.choices[0].message.content

print(ask("could you explain the best scheme that exist in this document"))

In [ ]:
while True:
    query = input("You: ")
    if query.lower() == "exit":
        break
    print("Bot:", ask(query))

In [ ]:
import json 
with open("chunks.json", "w") as f:
    json.dump(list_store, f)
    
faiss.write_index(ragindex, "index.faiss")

In [ ]:
with open("chunks.json", "r") as f:
    loaded_chunks = json.load(f)
loaded_index = faiss.read_index("index.faiss")
